In [ ]:
import boto3
import requests
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from urllib.parse import urlencode

In [ ]:
creds = boto3.Session(profile_name="default").get_credentials()

In [ ]:
REGION = "us-east-1"
BASE_URL = ""
STAGE = "dev"

### Sending Video for Prediction

In [ ]:
url = f"{BASE_URL}/{STAGE}/upload"

with open("patient_01_video.avi", "rb") as f:
    body = f.read()

headers = {"Content-Type": "video/x-msvideo"}

aws_req = AWSRequest(method="POST", url=url, data=body, headers=headers)
SigV4Auth(creds, "execute-api", REGION).add_auth(aws_req)

response = requests.post(url, data=body, headers=dict(aws_req.headers))
job_id = response.json()["id"]
print(response.json())

### Polling Prediction

In [ ]:
url = f"{BASE_URL}/{STAGE}/poll"   
url = f"{url}?{urlencode({'job_id': job_id})}"

aws_request = AWSRequest(method="GET", url=url)
SigV4Auth(creds, "execute-api", REGION).add_auth(aws_request)

response = requests.get(url, headers=dict(aws_request.headers))
print(response.json())

### Fetching Prediction

In [ ]:
url = f"{BASE_URL}/{STAGE}/fetch" 
url = f"{url}?{urlencode({'job_id': job_id})}"

aws_request = AWSRequest(method="GET", url=url)
SigV4Auth(creds, "execute-api", REGION).add_auth(aws_request)

response = requests.get(url, headers=dict(aws_request.headers))
print(response.json())